# Tutorial 06: Production Hardening (Evaluation & CI/CD)

We have successfully built a hardware-aware, self-correcting CRAG state machine. It runs efficiently on consumer GPUs and handles edge cases beautifully in our Jupyter notebooks.

However, a notebook is not a production system. If you deploy an LLM application without automated testing, containerization, and quantitative evaluation, you are flying blind. In this final tutorial, we will transform our research code into a deployable, enterprise-grade application.

### Prerequisites
* Completion of Tutorials 01 through 05.
* Basic understanding of Docker and Docker Compose.
* Familiarity with GitHub Actions.
* The `ragas` library installed (`pip install ragas`).


### Step 1: The Docker Mesh (Environment Parity)

"It works on my machine" is the enemy of production. To ensure our CRAG system runs identically on a local laptop and a cloud server, we wrap it in a Docker Compose mesh.

Instead of a single monolithic container, microservice architecture dictates we split our system into discrete components:

```yaml
# docker-compose.yaml (Conceptual Snippet)
version: '3.8'

services:
  # 1. The Vector Database (CPU bound)
  qdrant:
    image: qdrant/qdrant:latest
    ports:
      - "6333:6333"

  # 2. The LLM Provider (GPU bound)
  ollama:
    image: ollama/ollama:latest
    deploy:
      resources:
        reservations:
          devices:
            - driver: nvidia
              count: 1
              capabilities: [gpu]

  # 3. Our CRAG Application Layer
  rag_client:
    build: 
      context: ./client
    environment:
      - VECTOR_STORE_URL=http://qdrant:6333
      - LLM_HOST=http://ollama:11434
    depends_on:
      - qdrant
      - ollama
```

By decoupling the state machine (`rag_client`) from the inference engine (`ollama`) and the memory (`qdrant`), we can scale each component independently.


### Step 2: Deterministic Evaluation with Ragas

How do you know your RAG system is actually good? Eyeballing responses is subjective and unscalable. We need deterministic, offline metrics.

We will use the **Ragas** (Retrieval Augmented Generation Assessment) framework. Ragas uses an "LLM-as-a-Judge" paradigm to score our pipeline against a "Golden Dataset" (a set of predefined questions and perfect ground-truth answers).


In [ ]:
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy

print("--- Running Ragas Offline Evaluation ---")

# 1. Define our Golden Dataset
data_samples = {
    "question": ["What hardware handles generative inference?"],
    "answer": ["Generative inference is strictly fenced to the GPU."], # The system's output
    "contexts": [["The CRAG system utilizes Asymmetric Hardware Fencing. Generative inference is strictly fenced to the GPU."]], # What the retriever fetched
    "ground_truth": ["The GPU is responsible for generative inference."] # The perfect answer
}

dataset = Dataset.from_dict(data_samples)

# 2. Run the Evaluation Matrix
# Faithfulness: Did the answer hallucinate? (Based ONLY on contexts)
# Answer Relevancy: Did the answer actually address the user's question?
metrics = [faithfulness, answer_relevancy]
results = evaluate(dataset, metrics=metrics)

print(f"📊 Evaluation Results:\n{results}")


**Expected Output:**

```text
📊 Evaluation Results:
{'faithfulness': 1.0000, 'answer_relevancy': 0.9845}
```

---

### Step 3: CI/CD Quality Gates

Now that we have quantitative metrics, we can enforce quality gates. In a professional environment, no code should be merged into the `main` branch if it degrades the AI's performance.

We integrate our Ragas evaluation script into GitHub Actions. If a developer tweaks the prompt template or changes the embedding model, the CI/CD pipeline will automatically run the Ragas test suite:

```yaml
# .github/workflows/rag_eval.yml (Conceptual Snippet)
name: CRAG Quality Gate

on:
  pull_request:
    branches: [ main ]

jobs:
  evaluate_rag:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3
      
      - name: Install Dependencies
        run: pip install -r requirements.txt
        
      - name: Run Ragas Evaluation Suite
        run: pytest tests/test_ragas_eval.py --fail-under-relevancy=0.85
```

If the `answer_relevancy` drops below **0.85**, the pipeline fails, the pull request is blocked, and the hallucination never reaches production.

### The Architectural Win

This final tutorial separates junior prototypes from senior engineering:

1. **Reproducibility:** Anyone can clone the repository, run `docker-compose up`, and have the entire mesh running in 60 seconds.
2. **Accountability:** The system's accuracy is mathematically proven by Ragas.
3. **Safety:** The GitHub Action prevents regressions. You can confidently refactor your LangGraph state machine knowing the CI/CD pipeline will catch any degradation in response quality.

*Note: In this repository, see the production evaluation gate implementation in [`.github/workflows/rag_eval.yml`](../.github/workflows/rag_eval.yml) and [`tests/test_ragas_eval.py`](../tests/test_ragas_eval.py).*

---

### 🏋️ Challenge (Final Exercise)

Currently, our pipeline measures `faithfulness` and `answer_relevancy`. However, if our Vector Database is pulling 10 documents and the correct answer is buried at document #9, our latency and token costs will skyrocket.

**Your Task:**
Update the `tests/test_ragas_eval.py` file to include `context_precision` and `context_recall` from the Ragas library. These metrics evaluate the *retriever* rather than the *generator*. Set a CI/CD threshold ensuring `context_precision` remains above **0.80**.

If you have completed all six tutorials, you now possess a deep, practical understanding of how to architect, optimize, and deploy an enterprise-grade Corrective RAG system.
